In [1]:
pip install plotly


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import plotly.io as pio

In [3]:
lease = pd.read_csv("Leases.csv") 
price_availability_df = pd.read_csv("Price and Availability Data.csv")
occupancy_df = pd.read_csv("Major Market Occupancy Data-revised-20250412-051645.csv")

# 2\. Map of Top Flexible Office Markets

In a post-covid world, we want to identify U.S. office markets that are most suited to hybrid work and flexible leasing, using internal leasing, pricing, and occupancy data. (Are we still looking into external data like walkability or transit accesibility? Edwin?)

Question: Consider focusing on a specific indsutry? Or not? 

Key metrics: 

sublet_availability_proportion: 
avg_occupancy_proportion
overall_rent
availability_proportion

In [4]:
pio.renderers.default = 'notebook_connected'

# Aggregate by market
sublet_grouped = lease.groupby("market", as_index=False)["sublet_availability_proportion"].mean()
occupancy_grouped = occupancy_df.groupby("market", as_index=False)["avg_occupancy_proportion"].mean()
price_grouped = price_availability_df.groupby("market", as_index=False)[
    ["availability_proportion", "overall_rent"]
].mean()

# Merge data
merged_df = price_grouped.merge(sublet_grouped, on="market", how="left")
merged_df = merged_df.merge(occupancy_grouped, on="market", how="left")
print("Missing values:\n", merged_df.isna().sum())
print("Rows before cleaning:", merged_df.shape[0])

# Drop rows with missing critical data
merged_df.dropna(subset=[
    "availability_proportion",
    "sublet_availability_proportion",
    "overall_rent",
    "avg_occupancy_proportion"
], inplace=True)
print("Rows after cleaning:", merged_df.shape[0])

# Scale and score
scaler = MinMaxScaler()
scaled = scaler.fit_transform(merged_df[[
    "availability_proportion",
    "sublet_availability_proportion",
    "overall_rent",
    "avg_occupancy_proportion"
]])
scaled[:, 2] = 1 - scaled[:, 2]  # Invert rent (lower rent is better)

weights = np.array([0.25, 0.25, 0.25, 0.25])
merged_df["flexible_office_score"] = np.dot(scaled, weights)

# Come up with the score dataset
score_df = merged_df.sort_values(by="flexible_office_score", ascending=False)


Missing values:
 market                             0
availability_proportion            0
overall_rent                       0
sublet_availability_proportion     9
avg_occupancy_proportion          24
dtype: int64
Rows before cleaning: 30
Rows after cleaning: 6


In [5]:
%pip install geopy


[notice] A new release of pip is available: 23.0.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
city = {
    "New York": (40.730610, -73.935242),
    "San Francisco": (37.7749, -122.4194),
    "Chicago": (41.8781, -87.6298),
    "Atlanta": (33.7490, -84.3880),
    "Los Angeles": (34.0522, -118.2437),
    "Boston": (42.3601, -71.0589),
    "Houston": (29.7604, -95.3698),
    "Dallas": (32.7767, -96.7970),
    "Seattle": (47.6062, -122.3321),
    "Denver": (39.7392, -104.9903),
    "Austin": (30.2666, -97.7333),
    "Philadelphia": (39.9525, -75.165222),
    "Manhattan": (40.7766, -73.971321)
}
# from geopy import geocoders
# from geopy.geocoders import Nominatim

# gn = geocoders.GeoNames(username='hyperupcall')
# app = Nominatim(user_agent="tutorial")
# location = app.geocode("Nairobi, Kenya").raw
# print(location)
# def f1(x):
#     print(x)
#     return app.geocode(x).raw.lat

# def f2(x):
#     print(x)
#     return app.geocode(x).raw.lon

score_df1 = score_df.copy()
# score_df1 = merged_df.copy()
# merged_df

score_df1["lat"] = score_df1['market'].map(lambda x: city.get(x, (None, None))[0])
score_df1["lon"]= score_df1["market"].map(lambda x: city.get(x, (None, None))[1])
fig = px.scatter_geo(
    score_df1,
    lat="lat",
    lon="lon",
    # locations='market',
    size="flexible_office_score",
    color="flexible_office_score",
    hover_name="market",
    # projection="albers usa",
    # fitbounds="locations",  # auto-zoom
    title="Top U.S. Office Markets for Flexible Leasing",
    # size_max=25,
    # scope='usa'
    # color_continuous_scale="Viridis"
)
fig.update_layout(geo=dict(scope="usa"))
# # fig.update_traces(marker=dict(size=10, color='red'))

fig.show()

# # Save the map to an HTML file? Can't display here or render 
# # fig.write_html("flexible_office_map.html")


In [17]:
# Data cleaning
lease.columns = lease.columns.str.strip().str.lower()
price_availability_df.columns = price_availability_df.columns.str.strip().str.lower()
occupancy_df.columns = occupancy_df.columns.str.strip().str.lower()

# Step 1: Filter for tech
tech_leases = lease[lease["internal_industry"].str.contains("technology|it|information", case=False, na=False)]

# Step 2: Aggregate by market
sublet_grouped_tech = tech_leases.groupby("market", as_index=False)["sublet_availability_proportion"].mean()
price_grouped = price_availability_df.groupby("market", as_index=False)[["availability_proportion", "overall_rent"]].mean()
occupancy_grouped = occupancy_df.groupby("market", as_index=False)["avg_occupancy_proportion"].mean()

# Step 3: Merge all data
merged_df = price_grouped.merge(sublet_grouped_tech, on="market", how="left", indicator=True)
merged_df = merged_df.merge(occupancy_grouped, on="market", how="left")


# Step 4: Drop missing values
merged_df.dropna(subset=[
    "availability_proportion", "sublet_availability_proportion", 
    "overall_rent", "avg_occupancy_proportion"
], inplace=True)

# Step 5: Normalize and compute score
scaler = MinMaxScaler()
scaled = scaler.fit_transform(merged_df[[
    "availability_proportion",
    "sublet_availability_proportion",
    "overall_rent",
    "avg_occupancy_proportion"
]])
scaled[:, 2] = 1 - scaled[:, 2]  # invert rent

weights = np.array([0.25, 0.25, 0.25, 0.25])
merged_df["tech_flexible_office_score"] = np.dot(scaled, weights)


score_df_tech = merged_df.sort_values(by="tech_flexible_office_score", ascending=False)


In [19]:
score_df_tech.head(7)

,market,availability_proportion,overall_rent,sublet_availability_proportion,_merge,avg_occupancy_proportion,tech_flexible_office_score
10,Houston,0.278232,29.613804,0.021935,both,0.503773,0.773117
1,Austin,0.191850,42.048777,0.030032,both,0.517043,0.640919
11,Los Angeles,0.224983,44.143134,0.022468,both,0.398406,0.461200
22,San Francisco,0.233347,72.441176,0.039842,both,0.317613,0.433915
17,Philadelphia,0.203185,29.693244,0.019739,both,0.365611,0.418086
12,Manhattan,0.157688,77.627159,0.018549,both,0.349343,0.039776


In [23]:
city_coords = {
    "Houston": (29.7604, -95.3698),
    "Austin": (30.2672, -97.7431),
    "Los Angeles": (34.0522, -118.2437),
    "San Francisco": (37.7749, -122.4194),
    "Philadelphia": (39.9526, -75.1652),
    "Manhattan": (40.7128, -74.0060),
    "Seattle": (47.6062, -122.3321),
    "Boston": (42.3601, -71.0589),
    "Chicago": (41.8781, -87.6298),
    "Denver": (39.7392, -104.9903),
    "Atlanta": (33.7490, -84.3880),
    "Dallas": (32.7767, -96.7970),
    "San Diego": (32.7157, -117.1611),
    "San Jose": (37.3382, -121.8863),
    "Phoenix": (33.4484, -112.0740),
    "Miami": (25.7617, -80.1918),
    "Portland": (45.5051, -122.6750),
    "Minneapolis": (44.9778, -93.2650),
    "Raleigh": (35.7796, -78.6382),
    "Salt Lake City": (40.7608, -111.8910)
}
score_df_tech["lat"] = score_df_tech["market"].map(lambda x: city_coords.get(x, (None, None))[0])
score_df_tech["lon"] = score_df_tech["market"].map(lambda x: city_coords.get(x, (None, None))[1])
score_df_tech.dropna(subset=["lat", "lon"], inplace=True)

score_df_tech["score_label"] = score_df_tech["tech_flexible_office_score"].apply(lambda x: f"Score: {x:.2f}")


fig = px.scatter_geo(
    score_df_tech.head(15),
    lat="lat",
    lon="lon",
    size="tech_flexible_office_score",
    color="tech_flexible_office_score",
    hover_name="market",
    hover_data={
        "tech_flexible_office_score": ':.2f', 
        "lat": False,
        "lon": False
    },
    projection="albers usa",
    title="Top U.S. Office Markets for Flexible Leasing in the Tech Sector",
    size_max=25,
    color_continuous_scale="Viridis"
)

fig.update_layout(geo=dict(scope="usa"))
fig.show()
fig.write_html("tech_flexible_map.html")


In [9]:
score_df_tech["score_label"] = score_df_tech["tech_flexible_office_score"].apply(lambda x: f"Score: {x:.2f}")

fig = px.scatter_geo(
    score_df_tech.head(15),
    lat="lat",
    lon="lon",
    size="tech_flexible_office_score",
    color="tech_flexible_office_score",
    hover_name="market",
    hover_data={
        "score_label": True,
        "tech_flexible_office_score": False,
        "lat": False,
        "lon": False
    },
    projection="albers usa",
    title="Top U.S. Office Markets for Flexible Leasing in the Tech Sector",
    size_max=25,
    color_continuous_scale="Viridis"
)

fig.update_layout(
    geo=dict(scope="usa"),
    font=dict(family="Arial", size=14),
)
fig.update_coloraxes(colorbar_title="Flexibility Score")
fig.update_traces(marker=dict(line=dict(width=1, color='white')))

fig.show()
fig.write_html("tech_flexible_map_clean.html")

Show top 5 cities
Explain what drives their score (availability, sublet activity, affordability)

In [10]:
score_df_tech.head(10)

,market,availability_proportion,overall_rent,sublet_availability_proportion,avg_occupancy_proportion,tech_flexible_office_score,lat,lon,score_label
10,Houston,0.278232,29.613804,0.021935,0.503773,0.773117,29.7604,-95.3698,Score: 0.77
1,Austin,0.191850,42.048777,0.030032,0.517043,0.640919,30.2672,-97.7431,Score: 0.64
11,Los Angeles,0.224983,44.143134,0.022468,0.398406,0.461200,34.0522,-118.2437,Score: 0.46
22,San Francisco,0.233347,72.441176,0.039842,0.317613,0.433915,37.7749,-122.4194,Score: 0.43
17,Philadelphia,0.203185,29.693244,0.019739,0.365611,0.418086,39.9526,-75.1652,Score: 0.42


In [11]:
lease["market"].unique()

array(['Atlanta', 'Austin', 'Baltimore', 'Boston', 'Charlotte', 'Chicago',
       'Chicago Suburbs', 'Dallas/Ft Worth', 'Denver', 'Detroit',
       'Houston', 'Los Angeles', 'Manhattan', 'Nashville',
       'Northern New Jersey', 'Northern Virginia', 'Orange County',
       'Philadelphia', 'Phoenix', 'Raleigh/Durham', 'Salt Lake City',
       'San Diego', 'San Francisco', 'Seattle', 'South Bay/San Jose',
       'South Florida', 'Southern Maryland', 'Tampa', 'Washington D.C.'],
      dtype=object)

In [25]:
tech_leases = lease[lease["internal_industry"].str.contains("technology|it|information", case=False, na=False)]

sublet_grouped_tech = tech_leases.groupby("market", as_index=False)["sublet_availability_proportion"].mean()
price_grouped = price_availability_df.groupby("market", as_index=False)[["availability_proportion", "overall_rent"]].mean()
occupancy_grouped = occupancy_df.groupby("market", as_index=False)["avg_occupancy_proportion"].mean()

merged_df = price_grouped.merge(sublet_grouped_tech, on="market", how="left")
merged_df = merged_df.merge(occupancy_grouped, on="market", how="left")

merged_df.dropna(subset=[
    "availability_proportion", "sublet_availability_proportion",
    "overall_rent", "avg_occupancy_proportion"
], inplace=True)

scaler = MinMaxScaler()
scaled = scaler.fit_transform(merged_df[[
    "availability_proportion",
    "sublet_availability_proportion",
    "overall_rent",
    "avg_occupancy_proportion"
]])
scaled[:, 2] = 1 - scaled[:, 2]  # Invert rent

weights = np.array([0.25, 0.25, 0.25, 0.25])
merged_df["tech_flexible_office_score"] = np.dot(scaled, weights)

scorecard = merged_df.sort_values(by="tech_flexible_office_score", ascending=False)[[
    "market", "tech_flexible_office_score", "overall_rent", "sublet_availability_proportion"
]].copy()

scorecard["tech_flexible_office_score"] = scorecard["tech_flexible_office_score"].round(2)
scorecard["overall_rent"] = scorecard["overall_rent"].round(2)
scorecard["sublet_availability_proportion"] = (scorecard["sublet_availability_proportion"] * 100).round(1)


def recommend(row):
    if row["overall_rent"] < 30 and row["tech_flexible_office_score"] > 0.6:
        return "Startups"
    elif row["overall_rent"] < 50:
        return "Scaleups"
    else:
        return "Enterprise"

scorecard["recommended_for"] = scorecard.apply(recommend, axis=1)


print(scorecard.to_string(index=False))

       market  tech_flexible_office_score  overall_rent  sublet_availability_proportion recommended_for
      Houston                        0.77         29.61                             2.2        Startups
       Austin                        0.64         42.05                             3.0        Scaleups
  Los Angeles                        0.46         44.14                             2.2        Scaleups
San Francisco                        0.43         72.44                             4.0      Enterprise
 Philadelphia                        0.42         29.69                             2.0        Scaleups
    Manhattan                        0.04         77.63                             1.9      Enterprise


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4487cead-9e45-4286-b5b8-37614f9dfcb9' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>